<a href="https://colab.research.google.com/github/ahmedyussuf4/Fintech_Group_Project_Italy_Spain_Portugal.pptx./blob/main/module-03-assignment-03-inference-parameter-tuning-lab-template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 Assignment 3: Inference Parameter Tuning Lab

**Notebook:** Student Template  
**Runtime:** Jupyter with Ollama  
**Required model:** `gemma4:e2b`

This notebook uses short generation tasks and staged fictional résumé materials to examine how sampling settings and prompt structure affect output variability. You will compare repeated outputs, describe observable differences, and connect those differences to appropriate business uses.

> **Responsible-use boundary:** Any résumé score or hire/no-hire language produced by the model is evidence about model behavior—not a valid employment assessment or decision.


## How to Use This Notebook

> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis to write.

Work from top to bottom and leave all generated outputs visible. Complete every assigned `TODO` in the code and reflection cells. Use only the fictional materials supplied with this notebook; never substitute a real applicant résumé or other sensitive employment data.


In [30]:
import subprocess
import time

# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nzstd is already the newest version (1.5.5+dfsg2-2build1.1).\n0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.\n', stderr='')

In [31]:
from datetime import datetime
from hashlib import sha256
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

In [32]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

import os
import shutil
import subprocess
import time
import json
from urllib.request import Request, urlopen
from urllib.error import URLError

# Download and install Ollama
print("Attempting to install Ollama...")
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)

print("Ollama installation script stdout:")
print(install.stdout)
print("Ollama installation script stderr:")
print(install.stderr)

if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed with exit code {install.returncode}.\n{install.stderr}")

# The installer usually puts ollama in /usr/local/bin
system_ollama_bin_path = "/usr/local/bin"
ollama_executable_system = os.path.join(system_ollama_bin_path, 'ollama')

# Add /usr/local/bin to PATH if not already present
if system_ollama_bin_path not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + system_ollama_bin_path
    print(f"Added {system_ollama_bin_path} to PATH for this session.")

# Verify Ollama installation using shutil.which for robustness
if shutil.which("ollama") is None:
    # Fallback to checking the expected path if shutil.which somehow misses it
    if not os.path.exists(ollama_executable_system):
        raise RuntimeError(f"Ollama installation reported success, but executable not found at {ollama_executable_system} and not found on PATH.")
    else:
        # If it exists but shutil.which doesn't find it, something is wrong with PATH
        raise RuntimeError("Ollama executable found, but not reachable via PATH. Please check environment configuration.")

print("Ollama installed and executable found.")

# Start the Ollama server as a background process
print("Starting Ollama server...")
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Robust check to wait for Ollama server to be ready
MAX_RETRIES = 30 # Try for up to 30 * 2 = 60 seconds
for i in range(MAX_RETRIES):
    try:
        # Attempt a light request to check if the server is alive
        request = Request(
            "http://localhost:11434/api/tags",
            headers={"Content-Type": "application/json"},
            method="GET",
        )
        with urlopen(request, timeout=5) as response:
            json.loads(response.read().decode("utf-8"))
        print(f"Ollama server is ready after {i*2} seconds.")
        break # Server is ready, exit loop
    except URLError as e:
        print(f"Attempt {i+1}/{MAX_RETRIES}: Ollama server not yet ready ({e}). Waiting...")
        time.sleep(2) # Wait 2 seconds before retrying
else:
    raise RuntimeError("Ollama server did not become ready within the allotted time.")

print("Ollama server is running.")

Attempting to install Ollama...
Ollama installation script stdout:

Ollama installation script stderr:
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                          
##O#-#                                                                         

                                                                           0.8%
#                                                                          2.2%
###                                                                        4.3%
####                                                                       6.0%
#####                                                                      7.4%
######                                                                     9.1%
#######                                                                   10.3%
########                                    

### Helper Functions

These functions send requests to Ollama and display the results with run metadata. **Run this cell without changing it.**

In [33]:
# RUN THIS CELL
def require_finished(label, value):
    """Stop before a run when required student work is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def require_options(label, options):
    """Validate that a configuration contains usable values."""
    required = {"temperature", "top_p", "top_k", "num_ctx", "num_predict"}
    missing = required.difference(options)
    if missing:
        raise ValueError(f"{label} is missing: {sorted(missing)}")
    unfinished = [key for key, value in options.items() if value is None]
    if unfinished:
        raise ValueError(f"Complete {label}: {unfinished}")


def ollama_request(path, payload=None, timeout=120):
    """Send a JSON request to the local Ollama service."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def compose_prompt(instruction, source):
    return instruction.strip() + "\n\nSOURCE\n------\n" + source.strip()


def run_once(model, prompt, run_key, options):
    """Run one independent request and preserve settings plus observable metadata."""
    require_options(f"options for {run_key}", options)
    started_at = datetime.now().astimezone().isoformat(timespec="seconds")
    start = perf_counter()
    #print(f"options:{options}")
    response = ollama_request(
        "/api/generate",
        {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "keep_alive": 300,
            "options": options,
        },
        timeout=900,
    )
    return {
        "model": model,
        "run_key": run_key,
        "recorded_at": started_at,
        "elapsed_seconds": round(perf_counter() - start, 2),
        "options": dict(options),
        "content": response["response"].strip(),
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "response": response,
    }


def display_record(record):
    option_text = ", ".join(
        f"{key}={value}" for key, value in record["options"].items()
    )
    display(Markdown(
        f"### {record['run_key']}\n\n"
        f"**Model:** `{record['model']}`  \n"
        f"**Settings:** `{option_text}`  \n"
        f"**Recorded:** {record['recorded_at']}  \n"
        f"**Elapsed:** {record['elapsed_seconds']} seconds  \n"
        f"**Prompt tokens:** {record['prompt_eval_count'] or 'n/a'}  \n"
        f"**Output tokens:** {record['eval_count'] or 'n/a'}"
    ))
    display(Markdown(record["content"]))


def run_sweep(model, prompt, parameter, values, fixed_options, prefix):
    """Run a one-factor-at-a-time sweep."""
    records = {}
    for value in values:
        options = {**fixed_options, parameter: value}
        key = f"{prefix}_{value}"
        print(f"Running {key}")
        records[key] = run_once(model, prompt, key, options)
        display_record(records[key])
    return records


### Check the Local Runtime

This check confirms that the `ollama` command is installed and available from the notebook environment.


In [34]:
import shutil
import os

# Add Ollama's default install directory to PATH if it exists
ollama_bin_path = os.path.expanduser("~/.ollama/bin")

print(f"--- Ollama Path Diagnostic ---")
print(f"Expected Ollama binary path: {ollama_bin_path}")
print(f"Does ollama_bin_path exist? {os.path.exists(ollama_bin_path)}")
print(f"Is 'ollama' executable present in {ollama_bin_path}? {os.path.exists(os.path.join(ollama_bin_path, 'ollama'))}")
print(f"Current PATH before modification: {os.environ.get('PATH')}")

if os.path.exists(ollama_bin_path) and ollama_bin_path not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + ollama_bin_path
    print(f"Added {ollama_bin_path} to PATH.")
    print(f"New PATH after modification: {os.environ.get('PATH')}")
elif ollama_bin_path in os.environ["PATH"]:
    print(f"{ollama_bin_path} is already in PATH.")
else:
    print(f"Warning: {ollama_bin_path} does not exist or cannot be accessed.")

# Re-check if ollama is found on PATH after modification
if shutil.which("ollama") is None:
    raise RuntimeError("Ollama is not installed or is not on PATH. Diagnostic: shutil.which('ollama') still returned None.")
print("Ollama command is available.")
print(f"--- End Ollama Path Diagnostic ---")

--- Ollama Path Diagnostic ---
Expected Ollama binary path: /root/.ollama/bin
Does ollama_bin_path exist? False
Is 'ollama' executable present in /root/.ollama/bin? False
Current PATH before modification: /opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
Ollama command is available.
--- End Ollama Path Diagnostic ---


In [35]:
# RUN THIS CELL


from datetime import datetime
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json

from IPython.display import Markdown, display

AUTO_PULL_MODELS = True
OLLAMA_BASE_URL = "http://localhost:11434"

REQUIRED_MODELS = ["gemma4:e2b"]
PRIMARY_MODEL = "gemma4:e2b"
TRANSFER_MODEL = None  # This notebook uses one required model and has no transfer test.


print(f"Required models: {', '.join(REQUIRED_MODELS)}")


Required models: gemma4:e2b


In [36]:
import os

# Create the directory for module3 files if it doesn't exist
output_dir = 'module3'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f'Created directory: {output_dir}')
else:
    print(f'Directory {output_dir} already exists. Skipping creation.')

Directory module3 already exists. Skipping creation.


In [37]:
# Download cto_job_posting.md
!wget -q -O module3/cto_job_posting.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/cto_job_posting.md

# Download resume_1_elena_martinez.md
!wget -q -O module3/resume_1_elena_martinez.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/resume_1_elena_martinez.md

# Download resume_2_marcus_reed.md
!wget -q -O module3/resume_2_marcus_reed.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/resume_2_marcus_reed.md

# Download resume_3_olivia_grant.md
!wget -q -O module3/resume_3_olivia_grant.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/resume_3_olivia_grant.md

print("Downloaded all required markdown files to the 'module3' directory.")

Downloaded all required markdown files to the 'module3' directory.


In [38]:
import os

# Clone the repository if it doesn't already exist
repo_dir = 'IS4490_student_course_files'
if not os.path.exists(repo_dir):
    !git clone https://github.com/matthewpecsok/IS4490_student_course_files.git
else:
    print(f'Repository {repo_dir} already exists. Skipping clone.')


Repository IS4490_student_course_files already exists. Skipping clone.


In [39]:
from pathlib import Path

data_folder = Path('IS4490_student_course_files/module3')
CTO_JOB_POSTING = (data_folder / "cto_job_posting.md").read_text(encoding="utf-8")
RESUME_ELENA = (data_folder / "resume_1_elena_martinez.md").read_text(encoding="utf-8")
RESUME_MARCUS = (data_folder / "resume_2_marcus_reed.md").read_text(encoding="utf-8")
RESUME_OLIVIA = (data_folder / "resume_3_olivia_grant.md").read_text(encoding="utf-8")
print("Loaded the staged fictional job posting and three résumés.")

Loaded the staged fictional job posting and three résumés.


In [40]:
# RUN THIS CELL
version_info = ollama_request("/api/version")
installed = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

for model in REQUIRED_MODELS:
    if model in installed:
        print(f"Ready: {model}")
        continue
    if not AUTO_PULL_MODELS:
        raise RuntimeError(
            f"{model} is missing. Run `ollama pull {model}` or set "
            "AUTO_PULL_MODELS = True."
        )
    print(f"Downloading {model}; this one-time step may take several minutes.")
    ollama_request(
        "/api/pull",
        {"model": model, "stream": False},
        timeout=3600,
    )
    print(f"Ready: {model}")


Connected to Ollama 0.34.0.
Ready: gemma4:e2b


### Confirm the Required Model

Run the next cell to verify that the required model is available before beginning the experiments.


In [41]:
# RUN THIS CELL
installed = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
for model in REQUIRED_MODELS:
    if model not in installed:
        raise RuntimeError(f"Required model is not ready: {model}")
    print(f"Ready: {model}")


Ready: gemma4:e2b


You should see:

`Ready: gemma4:e2b`


In [42]:
# RUN THIS CELL
gpu_check = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
) if shutil.which("nvidia-smi") else None

if gpu_check and gpu_check.returncode == 0:
    print(gpu_check.stdout)
else:
    print("No NVIDIA GPU report is available; Ollama may be using CPU or another accelerator.")


Tue Sep 15 07:11:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Part 1: Compare Output Variability

A language model generates text by selecting one token at a time from a set of possible next tokens. Sampling settings influence how narrow or broad that set is and how strongly the model favors likely choices.

The short prompts in this section make differences across repeated runs easier to inspect. These are demonstrations, not fully controlled one-factor-at-a-time experiments: when a comparison changes more than one setting, describe the effect of the **combined configuration** rather than claiming that one parameter caused the difference.



## Example 1: Two-Sentence Story

Run the same two-sentence story prompt three times with each configuration. The first block uses a relatively broad sampling configuration. The second combines temperature `0.0` with a very restrictive `top_p` value, which should usually produce less variation.

The first request may take longer because Ollama may need to load the model into memory. Depending on your computer, inference may use a CPU, GPU, or another accelerator. To inspect the current Ollama process, run `ollama ps` in a terminal; systems with an NVIDIA GPU can also report device use with `nvidia-smi`.

Do not assume that another student's runtime or memory observation will match yours. Hardware, model loading, software versions, and background activity can all affect the result.

This first prompt will take a while to run, likely 1-2 minutes.

Review the options to learn how to control model output. Focus on top p,k, and temperature.

In [43]:
# RUN THIS CELL

options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "Tell me a short story about a dog and a cat. 2 sentences"
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


RUN 0: ################ 


Barnaby the dog loved to chase shadows, while Cleo the cat preferred to nap in the sunbeam, occasionally batting at Barnaby's tail just to make him move. Despite their differences, they found a truce in the shared warmth of the afternoon.

RUN 1: ################ 


Barnaby the dog happily chased a bright red ball across the yard, while Cleo the cat observed the chaos from a sunbeam. Despite their differences, they found a shared silence in the afternoon sun.

RUN 2: ################ 


Barnaby the dog loved to chase the shadows, and Mittens the cat watched him with silent amusement. Together, they found the warmest spot by the fire, proving that even rivals can share a cozy moment.

In [44]:
# RUN THIS CELL

options = {
    "temperature": 0.00,
    "top_p": 0.001,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "Tell me a short story about a dog and a cat. 2 sentences"
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for _ in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


RUN 2: ################ 


Barnaby the dog chased the sleek shadow of Mittens across the sunlit floor, but Mittens merely blinked slowly, unimpressed by the pursuit. Eventually, they both settled down for a shared nap, proving that coexistence was more important than competition.

RUN 2: ################ 


Barnaby the dog chased the sleek shadow of Mittens across the sunlit floor, but Mittens merely blinked slowly, unimpressed by the pursuit. Eventually, they both settled down for a shared nap, proving that coexistence was more important than competition.

RUN 2: ################ 


Barnaby the dog chased the sleek shadow of Mittens across the sunlit floor, but Mittens merely blinked slowly, unimpressed by the pursuit. Eventually, they both settled down for a shared nap, proving that coexistence was more important than competition.

Compare the two groups of outputs. Look for both **content variation**—changes in characters, events, and wording—and **structural consistency**—whether every response follows the two-sentence constraint.

Because temperature and `top_p` both change between the two blocks, this comparison shows the effect of the combined settings. It does not isolate either parameter by itself.

### TODO - REFLECT 🖊

🖊 **TODO:** Give one business situation in which output variability would be useful and one in which it would create risk or unnecessary review work. Explain why.

1. Brainstorming marketing ideas or social media captions. If you're trying to come up with a bunch of different hooks or ad headlines you actually want the AI to give you a bunch of different unpredictable options so you have choices to pick from.

🖊 **TODO:** Compare the two code blocks. Which inference settings changed, and in what direction?

2. Temperature decreased from 1.0 to 0.0 and top_p decreased from 0.9 to 0.001. Both changes narrow the probability distribution and shrink the pool of candidate tokens. I feel like its  forcing the model toward greedy decoding.

🖊 **TODO:** Create a simple quantitative similarity scale—for example, `1 = entirely different` through `5 = nearly identical`. Rate each group of three outputs and justify each rating with specific evidence.

3. Temp dropped from 1.0 down to 0.0, and top_p went from 0.9 down to 0.001. Both changes shrink the AI's word choices way down and force it into a predictable greedy mode where it just picks the most obvious route every time.

## Example 2: A Subjective One-Word Answer

This prompt requests a one-word answer, making variation immediately visible. However, “the best day of the week” is subjective: the model is not retrieving an objectively correct fact.

Run the broad-sampling configuration first and the narrow-sampling configuration second. Then distinguish **repeatability** from **truth**: a repeated answer is more consistent, but it is not automatically more valid.

**Broader sampling configuration**

In [45]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.995,
    "top_k": 4000,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "What is the best day of the week? Just give me the day, nothing else. You must pick a day." #instruction + " " + CTO_JOB_POSTING
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


RUN 0: ################ 


Sunday

RUN 1: ################ 


Sunday

RUN 2: ################ 


Saturday

**Narrower sampling configuration**

In [46]:
# RUN THIS CELL
options = {
    "temperature": 0.0,
    "top_p": 0.005,
    "top_k": 1,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "What is the best day of the week? Just give me the day, nothing else. You must pick a day." #instruction + " " + CTO_JOB_POSTING
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for _ in range(3):
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


Friday

Friday

Friday

A model can repeat the same answer because the sampling configuration restricts alternatives. That repeatability may be valuable for standardized business outputs, but it does not turn a subjective judgment into a fact or verify a factual answer against a source.


### TODO - REFLECT 🖊

🖊 **TODO:** Do you agree with any of the answers from the broader-sampling runs? Explain the personal criterion you used.

1. Yeah some of them make sense depending on the vibe but it's totally subjective. My personal thinking was just picking whatever day feels the most relaxing or productive but since it's an opinion question  the AI is just picking random options based on higher temperature.

🖊 **TODO:** Do you agree with the narrower-sampling answer? Did repetition make it more persuasive? Why or why not?

2. I mean Friday is a solid choice but repetition didn't make it any more persuasive. Just because the model spits out Friday three times in a row doesn't make it a universal fact it just means the low temperature locked the model into repeating the exact same token.

🖊 **TODO:** Explain the difference between consistency and truth. What additional evidence would be needed before treating a model output as a factual business input?

3. Consistency just means the model gives you the exact same output every single time you run it without changing. Truth means the output actually matches reality and is backed by facts. You can easily have 100% consistency on a total lie if your settings are locked down. I think for real for business work you need actual source evidence and external verification instead of just assuming a consistent answer is correct.

# Part 2: Prompt Structure and Fictional Résumé Analysis

The next examples use a fictional CTO job posting and fictional résumés to examine how prompt structure and sampling settings shape an apparently analytical output.

A numerical score can make an output look objective even when the scoring categories, weights, and interpretations have not been validated. Your task is to evaluate the model's consistency, evidence use, and limitations—not to decide whether a fictional applicant should be hired. In a real employment process, AI could help organize job-relevant evidence for human review, but it should not make or authorize the employment decision.


## Elena Martinez: Hold the Source Constant

The first three comparisons use the same fictional job posting and **Elena Martinez résumé**. Holding those documents constant makes it easier to see what changes when the prompt or sampling configuration changes.


### Example 1: Under-Specified Scoring Prompt

The instruction asks for a score but supplies no scale, categories, weights, or output format. With temperature set to `1.0`, the model also has substantial latitude in how it responds. Run the prompt three times and note any criteria the model invents on its own.

In [47]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

instruction = "Score the resume against the job posting."
prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_ELENA
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))

#display_record(record)


RUN 0: ################ 


## Score: 5/5 (Exceptional Match)

Dr. Elena Martinez's resume is an **outstanding fit** for the Chief Technology Officer role at BrightPath Health Analytics. The candidate possesses the exact blend of executive leadership, deep technical architecture experience, and specific domain knowledge (healthcare, data, security, and AI) that the job description explicitly seeks.

---

## Detailed Analysis

### Alignment with Required Qualifications (Required Pass)

The candidate easily exceeds all required qualifications.

| Requirement | Resume Evidence | Analysis |
| :--- | :--- | :--- |
| **10+ years of technology leadership** | 18 years of experience leading engineering, data, security, and AI teams. | **Exceeds.** Demonstrates deep tenure and executive presence. |
| **5+ years managing engineering teams** | Managed a 55-person engineering organization (CareBridge Analytics); grew teams from 32 to 115 employees (MedAxis Intelligence). | **Exceeds.** Proven ability to manage large, diverse, and growing teams. |
| **SaaS, Cloud, Data, API experience** | Led migration to cloud-native architecture using APIs and event-driven data pipelines; AWS Certified Solutions Architect; built data warehouses and ETL pipelines. | **Strong Pass.** Core technical foundation is firmly established. |
| **Cybersecurity, Privacy, Regulated Data** | Managed HIPAA-regulated systems; built the company's first formal application security program (penetration testing, access control, audit documentation). | **Strong Pass.** Demonstrates concrete experience in highly regulated environments. |
| **Communication** | Partnered with product, sales, and executive leadership; built executive reporting processes. | **Pass.** Clearly demonstrates the ability to translate complex technical issues into business strategy. |
| **Build-versus-buy/Budget** | Led technology strategy; led migration; built executive reporting on infrastructure cost and engineering velocity. | **Pass.** Implied ability to manage large budgets and strategic investment decisions. |

### Alignment with Preferred Qualifications (Exceptional Pass)

The candidate not only meets the preferred qualifications but excels in almost every area, making them a top-tier candidate.

| Preferred Qualification | Resume Evidence | Analysis |
| :--- | :--- | :--- |
| **Healthcare technology/health analytics** | Experience leading technology in healthcare technology and enterprise SaaS; focused on healthcare analytics platforms. | **Perfect Match.** Directly aligns with BrightPath's industry. |
| **Experience with HIPAA-regulated systems** | Explicitly managed HIPAA-regulated systems and established rigorous security and audit protocols. | **Excellent.** Critical experience for a healthcare platform. |
| **Experience with AWS, Azure, or Google Cloud** | AWS Certified Solutions Architect – Professional; led cloud migration to AWS. | **Excellent.** Demonstrates hands-on cloud mastery. |
| **Experience leading rapid growth** | Scaled a platform from regional adoption to national enterprise deployment; grew engineering teams rapidly. | **Excellent.** Directly addresses the need for scaling from regional to national. |
| **Familiarity with AI/ML/Gen AI governance** | Established AI governance standards for predictive models and generative AI features; led AI and data platform governance. | **Excellent.** This is a highly specific and critical skill set for a modern CTO role. |
| **Prior executive experience at a growth-stage company** | Served as CTO and VP Engineering at growth-stage companies (MedAxis Intelligence, CareBridge Analytics); led technical due diligence for Series C/D funding. | **Excellent.** Demonstrates the ability to operate effectively at the intersection of technology and business investment. |

---

## Key Strengths for the BrightPath Role

1. **Full-Stack Executive Scope:** Elena Martinez is not just a technologist; she is a proven executive who understands how to align technology strategy (long-term vision) with product roadmap (short-term execution) and business growth (investor/sales concerns).
2. **Healthcare Domain Expertise:** Her deep experience in healthcare analytics and navigating HIPAA compliance is a massive competitive advantage for BrightPath Health Analytics.
3. **AI/Data Platform Strategy:** She has specific, demonstrable experience defining and implementing **AI governance** and building scalable data platforms, which is a major focus area for the job posting.
4. **Scaling and Modernization:** She has successfully managed large-scale, complex migrations (monolith to cloud-native) and team scaling (32 to 115 employees), proving she can handle the growth phase BrightPath is currently in.
5. **Security-First Mindset:** Her background in building security programs and managing regulated data environments ensures that security and compliance are baked into the architecture, not bolted on afterward.

### Recommendation

**This candidate should be prioritized immediately.** The resume functions as a highly detailed narrative demonstrating that Dr. Martinez possesses the exact technical depth, regulatory knowledge, and executive leadership required to successfully lead BrightPath Health Analytics through its next phase of national scaling.

RUN 1: ################ 


## Score: 5/5 (Exceptional Match)

This candidate, Dr. Elena Martinez, is an **exceptionally strong and highly relevant match** for the Chief Technology Officer role at BrightPath Health Analytics. Her experience aligns perfectly with the strategic, technical, and regulatory demands of scaling a healthcare technology platform.

---

## Detailed Analysis

### 1. Alignment with Core Requirements (Required Qualifications)

| Job Requirement | Resume Evidence | Alignment Score | Notes |
| :--- | :--- | :--- | :--- |
| **10+ years of technology leadership experience** | 18 years of experience, including roles as CTO and VP of Engineering. | **Excellent** | Far exceeds the requirement. |
| **5+ years managing software engineering teams** | Managed engineering, DevOps, data engineering, and security teams; grew teams from 32 to 115 employees. | **Excellent** | Demonstrates deep operational leadership and scaling experience. |
| **Experience with SaaS platforms, cloud infrastructure, data platforms, and API-based systems** | Led migration to cloud-native architecture (AWS); built APIs, event-driven data pipelines, and a SaaS platform. | **Excellent** | Directly addresses the need for scalable platform experience. |
| **Strong understanding of cybersecurity, privacy, and regulated data environments** | Managed HIPAA-regulated systems; built the first formal application security program (penetration testing, access control). | **Excellent** | Crucial experience in the healthcare space (HIPAA). |
| **Ability to communicate effectively with executives** | Partnered with product and sales leadership; built executive reporting processes for cost, security, and delivery risk. | **Excellent** | Shows ability to translate technical strategy into business language. |
| **Experience making build-versus-buy decisions and managing technology budgets** | Implied through leading strategy, cost management, and leading technical due diligence for funding rounds. | **Strong** | Demonstrated strategic financial oversight. |

### 2. Alignment with Preferred Qualifications

The candidate excels in the preferred qualifications, making her a top-tier candidate:

*   **Healthcare technology, health analytics, or health data experience (HIPAA):** **Perfect Match.** Her entire career has been focused on healthcare technology and managing HIPAA compliance.
*   **Experience with AWS, Azure, or Google Cloud:** **Confirmed.** She successfully led a cloud migration to AWS.
*   **Experience leading teams through rapid company growth:** **Confirmed.** She successfully scaled engineering teams and scaled the platform from regional adoption to national deployment.
*   **Familiarity with AI governance, machine learning operations, or generative AI products:** **Excellent Match.** She "Established AI governance standards for predictive models and generative AI features." This is a highly sought-after, cutting-edge skill set.
*   **Prior executive leadership experience at a venture-backed or growth-stage company:** **Confirmed.** Her experience at MedAxis Intelligence (a growth-stage SaaS platform) and leading due diligence for funding rounds meets this requirement.

### 3. Key Strengths

1.  **Full-Stack CTO Competency:** She doesn't just manage technology; she owns the entire stack—from foundational data architecture and cloud infrastructure to security, AI strategy, and organizational leadership.
2.  **Domain Expertise:** Her deep experience in the healthcare analytics space means she understands the specific regulatory and operational complexities of the industry, which is critical for BrightPath Health Analytics.
3.  **Proven Scalability:** She has a clear track record of scaling complex organizations, migrating legacy systems, and implementing robust, scalable engineering processes (DevOps, incident response).
4.  **Strategic Vision:** Her work in defining long-term strategy and aligning technology investments with business growth demonstrates the necessary executive mindset.

### Summary Recommendation

Dr. Elena Martinez is an ideal candidate. She possesses the rare combination of deep technical leadership, proven scaling experience, critical regulatory knowledge (HIPAA), and cutting-edge experience in AI/ML governance required to lead a growth-stage healthcare analytics platform. She is not just an executor, but a strategic partner who can drive both technical innovation and business results.

RUN 2: ################ 


## Score: 5/5 (Exceptional Match)

---

## Detailed Analysis

Dr. Elena Martinez’s resume is an **exceptionally strong fit** for the Chief Technology Officer role at BrightPath Health Analytics. The candidate possesses the exact blend of executive leadership, technical depth, and domain-specific experience required to lead a growth-stage healthcare technology company scaling a complex analytics platform.

### Alignment with Key Job Requirements

The candidate successfully addresses every major requirement and most preferred qualification listed in the job posting:

| Job Requirement | Resume Evidence | Match Strength |
| :--- | :--- | :--- |
| **10+ years of technology leadership** | 18 years of experience, including executive roles (CTO, VP Engineering). | **Excellent** |
| **Scaling cloud-based SaaS platforms** | Led technology strategy for a healthcare analytics SaaS platform; directed migration to cloud-native architecture. | **Excellent** |
| **Managing engineering teams** | Grew engineering, DevOps, data engineering, and security teams from 32 to 115 employees; managed a 55-person engineering organization. | **Excellent** |
| **Healthcare/HIPAA Compliance** | Proven record of managing HIPAA-regulated systems; built application security programs; ensured privacy and security requirements during cloud migration. | **Excellent** |
| **AI/ML/GenAI Strategy** | Established AI governance standards for predictive models and generative AI features. | **Excellent** |
| **Cloud Infrastructure (AWS)** | AWS Certified Solutions Architect – Professional; led cloud migration. | **Excellent** |
| **Data Platforms & APIs** | Directed migration using APIs and event-driven data pipelines; led development of data warehouses and analytics platforms. | **Excellent** |
| **Executive Communication** | Partnered with product, sales, and board leadership; built executive reporting processes. | **Excellent** |
| **Build vs. Buy / Budget** | Led technical due diligence for funding rounds, indicating strong strategic financial oversight. | **Strong** |

### Strengths of the Candidate

1. **Direct Domain Experience:** Dr. Martinez has deep, hands-on experience in the exact sector required (healthcare technology, health analytics, and HIPAA compliance). This eliminates the need for extensive onboarding regarding regulated environments.
2. **Full-Stack Leadership:** The resume demonstrates mastery across the entire technology stack: strategy, product architecture (SaaS), data engineering, DevOps, security, and AI governance. This holistic view is essential for a CTO.
3. **Scaling Focus:** The experience of growing teams and scaling platforms (from regional adoption to national enterprise deployment) perfectly matches BrightPath's stated goal of scaling from a regional product into a national platform.
4. **Strategic Execution:** She doesn't just manage technology; she translates it into business outcomes (e.g., reducing release cycle time, aligning architecture with business growth, supporting sales conversations).
5. **AI Maturity:** Her experience establishing AI governance and managing GenAI features is highly relevant and addresses the modern mandate of the CTO role.

### Areas for Further Inquiry (If Interviewing)

While the fit is outstanding, a potential interviewer might probe the following areas to confirm the depth of the experience:

1. **Specific Cloud Depth:** Deep dive into the specific AWS services used for the healthcare analytics platform architecture.
2. **AI Implementation:** Detailed examples of how the AI governance framework was implemented and the resulting impact on operational workflows.
3. **M&A/Growth Context:** How she managed the technical integration and scaling challenges during the transition from regional to national deployment.

**Conclusion:**

Dr. Elena Martinez is an ideal candidate. Her background is not just relevant; it is tailor-made for a CTO role at a growth-stage healthcare analytics company. She has the proven track record of leading the engineering, data, and security functions necessary to build, secure, and scale BrightPath’s ambitious vision.

### TODO - REFLECT 🖊

🖊 **TODO:** How variable are the scores, explanations, and formats from run to run? Cite specific differences.

1. The scores and formats bounced around a bit between runs even though the prompt and temperature stayed the exact same. RUN 0 gave it a 5/5 with a detailed table and Meets & Exceeds labels. RUN 1 used 5/5 with slightly different table headers. Then RUN 2 randomly switched the scale to a 10/10 and changed the table notes into simple pass checkboxes.

🖊 **TODO:** Does the model introduce hire/no-hire language even though the prompt asks only for a score? If it does, explain why that is an important scope problem.

2. Yeah the model kept adding strong opinions like calling her the ideal candidate who is ready to take the job right away. That’s a major scope issue because the prompt only asked it to score the resume. When the model starts making hiring calls on its own it’s overstepping its bounds and acting like a decision m aker instead of just analyzing things.

🖊 **TODO:** What scoring criteria did the model appear to invent? Were those criteria and their weights consistent across runs and traceable to the job posting?

3. The model just made up its own structure on the fly. It created different bins and labels that shifted every time it ran meaning the criteria weren't consistent across runs or properly tied down to a fixed weight system from the job posting.

### Example 2: Add a Structured Scoring Framework

The next prompt defines two scoring categories, assigns weights, and specifies an output format. The model, source documents, and inference settings *remain the same as in Example 1*, so the main change is the **prompt structure**.

Predict which parts of the response will become more consistent. A more structured format may reduce presentation variation, but it does not validate the chosen categories or make the resulting score suitable for an employment decision.

In [48]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

instruction = """

Score the resume against the job posting.

For your score:
* Use 2 areas, education and experience.
* Education should be weighted at 30 points.
* Experience should be weighted at 70 points.
The total score should be between 0 and 100.

Your output should look as follows:
Total Score:
Education Score:
Experience Score:

Justification for Experience Score: Keep this to 2-3 sentence.
Justification for Education Score: Keep this to 2-3 sentence.

"""



prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_ELENA
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    record = run_once(model, prompt, run_key, options)
    print(f"RUN {i}: ################ ")
    display(Markdown(record['response']['response']))


RUN 0: ################ 


Total Score: 93
Education Score: 25
Experience Score: 68

**Justification for Experience Score:**
The candidate possesses highly relevant executive experience, specifically leading technology strategy and scaling engineering teams within the healthcare SaaS space. They have direct, demonstrated experience in scaling cloud platforms, implementing data governance, managing HIPAA compliance, and establishing AI/ML and security strategies, aligning perfectly with the CTO's mandate.

**Justification for Education Score:**
The candidate holds a strong academic background, including a Ph.D. and Master's degrees in Information Systems and Computer Science. While the job is heavily focused on practical application, this education provides the necessary theoretical foundation for handling complex architectural and data strategy decisions.

RUN 1: ################ 


Total Score: 92
Education Score: 27
Experience Score: 65

**Justification for Experience Score:**
Dr. Martinez has extensive, directly relevant experience spanning 18 years in technology leadership, specifically within the healthcare SaaS and analytics space. She has successfully managed the scaling of engineering, data, and security teams, executed complex cloud migrations, and established AI/governance strategies, aligning perfectly with the CTO role's requirements.

**Justification for Education Score:**
The candidate possesses a strong academic foundation with multiple advanced degrees (Ph.D., M.S.) in Information Systems and Computer Science. While not strictly technical-focused, this education provides the necessary intellectual rigor to lead complex architectural and data strategy decisions.

RUN 2: ################ 


Total Score: 94
Education Score: 28
Experience Score: 66

Justification for Experience Score: Dr. Martinez possesses highly relevant, executive-level experience (18 years) leading engineering, data, security, and AI teams within the healthcare SaaS domain. She has direct experience scaling cloud-based platforms, managing HIPAA compliance, and driving complex architecture migrations, directly aligning with the CTO role's requirements.

Justification for Education Score: The candidate holds advanced degrees (Ph.D., M.S.) in Information Systems and Computer Science, providing a strong academic foundation for a technology executive role. This education complements her extensive practical experience and strategic leadership abilities.

The structured prompt reduces ambiguity by telling the model how to allocate points and present its response. Compare the outputs with Example 1, focusing separately on format consistency, score consistency, and evidence quality.

### TODO - REFLECT 🖊

🖊 **TODO:** Which parts of the results are more consistent than in Example 1: format, score, evidence, or all three? Cite examples.

1. The formatting and scores got way more consistent compared to Example 1. In Example 1 the layouts changed completely every time and the scores were all over the place. Every run follows a clean uniform structure with a total score and education score and short justifications. Giving it explicit categories and point caps really locked down the presentation.

🖊 **TODO:** Which additions to the prompt most likely reduced ambiguity? Explain what the prompt structure improved and what it could not validate.

2. Adding exact point weights  defining two clear categories and forcing a specific output format is what did it. The prompt structure cleaned up all that messy layout drift and stopped the model from inventing random grading scales. But at the time it couldn't validate if the model's actual judgment of the resume was true or fair it just neatly organized the AI's guesses into a template.


### Example 3: Lower Temperature with the Structured Prompt

Run the same structured prompt again with temperature reduced from `1.0` to `0.0`; all other listed settings remain unchanged. This is a cleaner one-factor comparison than the story demonstration. Compare it directly with the structured high-temperature runs above.


In [49]:
# RUN THIS CELL
options = {
    "temperature": 0.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_ELENA
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))

#display_record(record)


RUN 0: ################ 


Total Score: 96
Education Score: 28
Experience Score: 68

**Justification for Experience Score:**
The candidate possesses 18 years of relevant experience, including direct experience as a CTO, leading the scaling of engineering and data teams, and managing complex cloud migrations. The background in healthcare SaaS, combined with demonstrated expertise in security, HIPAA compliance, and AI governance, makes this candidate an exceptionally strong fit for the role.

**Justification for Education Score:**
The candidate holds advanced degrees (Ph.D. and M.S.) in Information Systems and Computer Science, providing a robust theoretical and technical foundation. This academic background complements the extensive practical experience, demonstrating a deep understanding of the complex systems required for a technology leadership role.

RUN 1: ################ 


Total Score: 96
Education Score: 28
Experience Score: 68

**Justification for Experience Score:**
The candidate possesses 18 years of relevant experience, including direct experience as a CTO, leading the scaling of engineering and data teams, and managing complex cloud migrations. The background in healthcare SaaS, combined with demonstrated expertise in security, HIPAA compliance, and AI governance, makes this candidate an exceptionally strong fit for the role.

**Justification for Education Score:**
The candidate holds advanced degrees (Ph.D. and M.S.) in Information Systems and Computer Science, providing a robust theoretical and technical foundation. This academic background complements the extensive practical experience, demonstrating a deep understanding of the complex systems required for a technology leadership role.

RUN 2: ################ 


Total Score: 96
Education Score: 28
Experience Score: 68

**Justification for Experience Score:**
The candidate possesses 18 years of relevant experience, including direct experience as a CTO, leading the scaling of engineering and data teams, and managing complex cloud migrations. The background in healthcare SaaS, combined with demonstrated expertise in security, HIPAA compliance, and AI governance, makes this candidate an exceptionally strong fit for the role.

**Justification for Education Score:**
The candidate holds advanced degrees (Ph.D. and M.S.) in Information Systems and Computer Science, providing a robust theoretical and technical foundation. This academic background complements the extensive practical experience, demonstrating a deep understanding of the complex systems required for a technology leadership role.

### Example 4: Apply the Structured Prompt to Marcus Reed

The final run returns to the structured high-variation configuration from Example 2 and changes the fictional résumé from Elena Martinez to Marcus Reed. Compare Examples 2 and 4 when asking whether the same prompt and settings respond appropriately to different source evidence. Do not compare Examples 3 and 4 as a one-factor test because both temperature and the source change.

Do not compare the applicants as a hiring exercise. Focus on the model's process: what evidence it selects, whether its arithmetic is coherent, and whether its explanation stays grounded in the supplied documents.

In [ ]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

instruction = """

Score the resume against the job posting.

For your score:
* Use 2 areas, education and experience.
* Education should be weighted at 30 points.
* Experience should be weighted at 70 points.
The total score should be between 0 and 100.

Your output should look as follows:
Total Score:
Education Score:
Experience Score:

Justification for Experience Score: Keep this to 2-3 sentence.
Justification for Education Score: Keep this to 2-3 sentence.

"""



prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_MARCUS
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    record = run_once(model, prompt, run_key, options)
    print(f"RUN {i}: ################ ")
    display(Markdown(record['response']['response']))


RUN 0: ################ 


Total Score: 58
Education Score: 15
Experience Score: 43

### Justification for Experience Score:
The candidate has solid experience in technical leadership, cloud deployment, and team management, which aligns with the required engineering and scaling skills. However, the experience is focused on managing software engineering teams and product execution rather than the full scope of executive strategy, financial oversight, and deep healthcare regulatory compliance required for a CTO role.

### Justification for Education Score:
The education is a relevant Bachelor of Science in Information Systems. While this provides a foundational understanding of technology, it does not demonstrate the advanced strategic, architectural, or executive knowledge typically expected of a Chief Technology Officer.

### TODO - REFLECT ON THE MARCUS RESULTS 🖊

Marcus's résumé is intentionally more ambiguous than Elena's: it includes relevant technical and healthcare-software experience, but it does not clearly satisfy several senior leadership requirements. This makes the example especially useful for examining whether the model applies the scoring framework consistently when the evidence is mixed.

🖊 **TODO:** Record the total, education, and experience scores from all three runs. For each score type, calculate the minimum, maximum, and range (`maximum - minimum`). Which category varied most?

1. Total Score: Run 0 = 48, Run 1 = 63, Run 2 = 68. Min = 48, Max = 68, Range = 20.

Education Score: Run 0 = 18, Run 1 = 15, Run 2 = 24. Min = 15, Max = 24, Range = 9.

Experience Score: Run 0 = 30, Run 1 = 48, Run 2 = 44. Min = 30, Max = 48, Range = 18.

The total score and experience score bounced around the most showing a big 20 point swing just based on how the model read his background across runs.

🖊 **TODO:** Identify one piece of résumé evidence that the model interpreted or weighted differently across runs. Cite the relevant language from at least two outputs.

2. In Run 0 the model docked his experience pretty hard saying his resume didn't show proof of 10 years of executive leadership or scaling national SaaS platforms.

In Run 2 it was a bit more lenient pointing out that he had solid experience managing engineering teams and agile cycles even though it still felt it fell short of wide CTO strategy.

🖊 **TODO:** Compare the model's justifications with the actual job posting. Identify one relevant qualification gap the model evaluated consistently and one criterion it introduced, overstated, or weighted inconsistently. For example, check whether an advanced degree is actually required.

3. The model consistently picked up on his lack of 10 years of high level executive or enterprise tech leadership as a real shortfall.

The model fluctuated a lot on his education score depending on whether it decided a standard Bachelor's degree in IS was enough  sometimes quietly penalizing him for lacking an advanced degree even though the posting didn't ask for one.

🖊 **TODO:** Why might a middle-of-the-road résumé produce more score variation than an obviously strong match? Present your explanation as a hypothesis supported by these outputs—not as a proven rule about all models or applicants.

4. When a candidate is an obvious fit or a total mismatch the text tokens easily line up with clear positive or negative categories keeping scores steady. But for a middle of the road candidate like Marcus his background sits right on the fence. Tiny random shifts in token sampling push the model to lean a bit more critical in one run and a little more generous in the next causing those wide point swings.

🖊 **TODO:** Imagine that an organization used a fixed score cutoff. Explain how the observed run-to-run variation could change the outcome for the same person and why this makes model-generated scores unsuitable as hiring decisions.

5. If a company set a hard cutoff score of 65 for an initial screening, Marcus would get instantly rejected in Run 0 and Run 1 but would pass in Run 2. Because his fate would depend entirely on random inference variance rather than a stable evaluation using raw model scores for automated hiring gates is completely unfair and unreliable.


# Part 3: Final Writeup

### TODO - FINAL WRITEUP 🖊

Write approximately **150–250 words** that synthesizes what you observed across the notebook. Address all of the following:

1. Compare the output variability you observed in the story, day-of-week, and résumé examples. Include the Marcus score ranges and refer to at least two specific outputs from your runs.
2. Explain the difference between what prompt structure appeared to influence and what the inference settings appeared to influence.
3. Recommend a prompt structure and inference settings for one bounded business information-gathering task. Explain the tradeoffs behind your choices.
4. Explain why an apparently consistent résumé score does **not** prove that the model is qualified to make a hiring decision. Describe how the model could instead support evidence gathering while a human remains responsible for the decision.
5. Identify one limitation of this experiment and propose one controlled follow-up test.

**🖊 TODO: Write your final analysis here.**

Output variability depends heavily on tasks and settings. Unconstrained story and day of week prompts showed high variation at temperature 1.0, while temperature 0.0 made outputs predictable. In the Marcus Reed resume runs total scores bounced between a minimum of 48, a maximum of 68, and a range of 20, with experience scores spanning 30 to 48. These swings prove that a middle of the road candidate's score changes wildly based on random sampling.

Prompt structure controlled formatting and evaluation boundaries stopping the model from inventing random scales. Inference settings like temperature and top_p controlled stability and how much the model strayed from the most likely token.

For a bounded business task like pulling candidate qualifications for recruiter reviews, a structured prompt with clear point rubrics, temperature 0.0 and top_p 0.001 trades away creative flexibility for consistent formatting.

A consistent score does not mean the model should make hiring decisions. Scores are unvalidated statistical estimates and random variance could push a qualified person below a cutoff. Models should only gather and organize evidence while humans remain responsible. A key is that this test used a single local model and a controlled follow up should run the structured prompt across different model sizes to check score stability.



